In [ ]:
"""
03_evaluation_analysis.ipynb
==============================
Copy each section between # %% markers into separate notebook cells.
Update paths to your actual directories.
"""

# 03 - Evaluation Analysis

In [6]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO

# ─── UPDATE THESE PATHS ───
MODEL_PATH = "../training_logs/exp_001_yolo26s_full_finetune/weights/best.pt"
DATA_YAML = r"C:\Users\rajcr\Desktop\Study\MTech\Courses\CV\Project\Dataset\leaked_data_split\data.yaml"
EVAL_RESULTS_DIR = "../evaluation_results"   # if evaluate.py was already run
# ───────────────────────────

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

COLORS = {
    "bike": "#2196F3",
    "helmet": "#4CAF50",
    "no-helmet": "#F44336",
    "number-plate": "#FF9800",
}

SAVE_DIR = os.path.join(EVAL_RESULTS_DIR, "analysis_plots")
os.makedirs(SAVE_DIR, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(os.path.join(SAVE_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    print(f"Saved: {name}.png")

In [7]:
model = YOLO(MODEL_PATH)
metrics = model.val(
    data=DATA_YAML,
    split="test",
    conf=0.25,
    iou=0.5,
    plots=True,
    verbose=True,
)

print(f"\nmAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

# Per-class
print("\nPer-class AP@50:")
for i, name in model.names.items():
    try:
        print(f"  {name:15s}: {metrics.box.ap50[i]:.4f}")
    except IndexError:
        pass

Ultralytics 8.4.50  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO26s summary (fused): 122 layers, 9,466,728 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 15.217.1 MB/s, size: 79.0 KB)
val: Scanning C:\Users\rajcr\Desktop\Study\MTech\Courses\CV\Project\Dataset\leaked_data_split\test\labels... 653 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 653/653 619.3it/s 1.1s0.0s
val: New cache created: C:\Users\rajcr\Desktop\Study\MTech\Courses\CV\Project\Dataset\leaked_data_split\test\labels.cache
WARNING Box and segment counts should be equal, but got len(segments) = 23, len(boxes) = 3061. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 3.5s/it 2:227.4sss
                

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

class_names = [model.names[i] for i in sorted(model.names.keys())]
class_colors = [COLORS.get(n, "#999999") for n in class_names]

# AP@50
ax = axes[0]
ap50_vals = [float(metrics.box.ap50[i]) for i in range(len(class_names))]
ax.bar(class_names, ap50_vals, color=class_colors)
ax.set_title("AP@50 Per Class")
ax.set_ylabel("AP@50")
ax.set_ylim(0, 1)
for i, v in enumerate(ap50_vals):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

# AP@50-95
ax = axes[1]
ap5095_vals = [float(metrics.box.maps[i]) for i in range(len(class_names))]
ax.bar(class_names, ap5095_vals, color=class_colors)
ax.set_title("AP@50-95 Per Class")
ax.set_ylabel("AP@50-95")
ax.set_ylim(0, 1)
for i, v in enumerate(ap5095_vals):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)

# Combined P and R
ax = axes[2]
x = np.arange(len(class_names))
width = 0.35
try:
    p_vals = [float(metrics.box.p[i]) for i in range(len(class_names))]
    r_vals = [float(metrics.box.r[i]) for i in range(len(class_names))]
    ax.bar(x - width/2, p_vals, width, label="Precision", color="#1976D2", alpha=0.8)
    ax.bar(x + width/2, r_vals, width, label="Recall", color="#D32F2F", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.set_title("Precision & Recall Per Class")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.legend()
except Exception:
    ax.set_title("Precision & Recall (not available)")

fig.suptitle("Test Set Performance", fontsize=16)
fig.tight_layout()
save_fig(fig, "01_per_class_metrics")
plt.show()

Saved: 01_per_class_metrics.png


<Figure size 1500x500 with 3 Axes>

In [10]:
conf_thresholds = np.arange(0.1, 0.9, 0.05)
map50_scores = []
precision_scores = []
recall_scores = []

print("Running confidence threshold sweep...")
for conf in conf_thresholds:
    m = model.val(data=DATA_YAML, split="test", conf=conf, iou=0.5, verbose=False, plots=False)
    map50_scores.append(float(m.box.map50))
    precision_scores.append(float(m.box.mp))
    recall_scores.append(float(m.box.mr))
    print(f"  conf={conf:.2f}: mAP50={map50_scores[-1]:.4f}, P={precision_scores[-1]:.4f}, R={recall_scores[-1]:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(conf_thresholds, map50_scores, label="mAP@50", color="#4CAF50", linewidth=2)
ax.plot(conf_thresholds, precision_scores, label="Precision", color="#1976D2", linewidth=2)
ax.plot(conf_thresholds, recall_scores, label="Recall", color="#D32F2F", linewidth=2)

# Mark optimal conf for mAP
best_idx = np.argmax(map50_scores)
best_conf = conf_thresholds[best_idx]
ax.axvline(best_conf, color="gray", linestyle=":", alpha=0.7,
           label=f"Best conf={best_conf:.2f}")

ax.set_xlabel("Confidence Threshold")
ax.set_ylabel("Score")
ax.set_title("Confidence Threshold Sensitivity")
ax.legend()
fig.tight_layout()
save_fig(fig, "02_confidence_sensitivity")
plt.show()

print(f"\nOptimal confidence threshold: {best_conf:.2f} (mAP@50={map50_scores[best_idx]:.4f})")

Running confidence threshold sweep...
Ultralytics 8.4.50  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 420.5440.0 MB/s, size: 193.8 KB)
val: Scanning C:\Users\rajcr\Desktop\Study\MTech\Courses\CV\Project\Dataset\leaked_data_split\test\labels.cache... 653 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 653/653  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 23, len(boxes) = 3061. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 10.2s/it 7:0015.3s
                   all        653       3061      0.899      0.891      0.918      0.692
Speed: 15.8ms preprocess, 622.4ms inference, 0.0ms loss, 0.4ms postprocess per image
  conf=0.10: m

KeyboardInterrupt: 

In [ ]:
# Run prediction on test images to get confidence values
import glob

test_images_dir = os.path.join(os.path.dirname(DATA_YAML), "test", "images")
test_images = sorted(glob.glob(os.path.join(test_images_dir, "*.jpg")))

all_confs = defaultdict(list)

for img_path in test_images[:200]:  # limit for speed
    results = model(img_path, conf=0.1, verbose=False)
    if results and results[0].boxes is not None:
        for i in range(len(results[0].boxes)):
            cls = int(results[0].boxes.cls[i])
            conf = float(results[0].boxes.conf[i])
            cls_name = model.names[cls]
            all_confs[cls_name].append(conf)

fig, ax = plt.subplots(figsize=(10, 6))
for cls_name, color in COLORS.items():
    if cls_name in all_confs:
        ax.hist(all_confs[cls_name], bins=30, alpha=0.5, color=color,
                label=f"{cls_name} (n={len(all_confs[cls_name])})", edgecolor="black")

ax.axvline(0.25, color="red", linestyle="--", label="Current threshold (0.25)")
ax.set_xlabel("Detection Confidence")
ax.set_ylabel("Count")
ax.set_title("Detection Confidence Distribution by Class")
ax.legend()
fig.tight_layout()
save_fig(fig, "03_confidence_distribution")
plt.show()

In [11]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

all_sizes = defaultdict(list)
all_size_confs = defaultdict(list)

for img_path in test_images[:200]:
    img = cv2.imread(img_path)
    if img is None:
        continue
    h, w = img.shape[:2]

    results = model(img_path, conf=0.1, verbose=False)
    if results and results[0].boxes is not None:
        boxes = results[0].boxes
        for i in range(len(boxes)):
            cls = int(boxes.cls[i])
            conf = float(boxes.conf[i])
            bbox = boxes.xyxy[i].cpu().numpy()
            area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
            cls_name = model.names[cls]
            all_sizes[cls_name].append(area)
            all_size_confs[cls_name].append(conf)

for idx, cls_name in enumerate(COLORS.keys()):
    ax = axes[idx // 2][idx % 2]
    if cls_name in all_sizes:
        ax.scatter(all_sizes[cls_name], all_size_confs[cls_name],
                   alpha=0.3, s=10, color=COLORS[cls_name])
        ax.set_title(f"{cls_name} (n={len(all_sizes[cls_name])})")
    else:
        ax.set_title(f"{cls_name} — no detections")
    ax.set_xlabel("Bbox Area (px²)")
    ax.set_ylabel("Confidence")
    ax.axhline(0.25, color="red", linestyle="--", alpha=0.5)

fig.suptitle("Detection Size vs Confidence", fontsize=14)
fig.tight_layout()
save_fig(fig, "04_size_vs_confidence")
plt.show()

NameError: name 'test_images' is not defined

In [12]:
report_path = os.path.join(EVAL_RESULTS_DIR, "eval_report.json")
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)

    print("=" * 60)
    print("  PIPELINE EVALUATION REPORT")
    print("=" * 60)

    m = report["metrics"]
    print(f"\n  Violation detection rate:  {m['violation_detection_rate']:.4f}")
    print(f"  Rider count accuracy:      {m['rider_count_accuracy']:.4f}")
    print(f"  Helmet count accuracy:     {m['helmet_count_accuracy']:.4f}")
    print(f"  OCR exact match:           {m['ocr_exact_match']:.4f}")
    print(f"  OCR edit distance:         {m['ocr_edit_distance_avg']:.4f}")
    print(f"  FINAL SCORE:               {m['final_score']:.4f}")

    # Failure distribution
    if "failure_counts" in report:
        print("\n  Failure distribution:")
        for cat, count in report["failure_counts"].items():
            print(f"    {cat:25s}: {count}")
else:
    print("Pipeline evaluation report not found. Run evaluate.py first for full metrics.")

  PIPELINE EVALUATION REPORT

  Violation detection rate:  0.5862
  Rider count accuracy:      0.4138
  Helmet count accuracy:     0.5172
  OCR exact match:           0.3103
  OCR edit distance:         0.6360
  FINAL SCORE:               0.5508

  Failure distribution:
    missed_violation         : 10
    false_positive           : 3
    wrong_rider_count        : 5
    wrong_helmet_count       : 2
    ocr_failure              : 7


In [13]:
print("\n" + "=" * 60)
print("  TEST EVALUATION SUMMARY")
print("=" * 60)
print(f"  mAP@50:          {metrics.box.map50:.4f}")
print(f"  mAP@50-95:       {metrics.box.map:.4f}")
print(f"  Precision:        {metrics.box.mp:.4f}")
print(f"  Recall:           {metrics.box.mr:.4f}")
print(f"  Optimal conf:     {best_conf:.2f}")
print(f"\n  Per-class AP@50:")
for i, name in model.names.items():
    try:
        print(f"    {name:15s}: {metrics.box.ap50[i]:.4f}")
    except IndexError:
        pass


  TEST EVALUATION SUMMARY
  mAP@50:          0.8972
  mAP@50-95:       0.6793
  Precision:        0.8985
  Recall:           0.8913


NameError: name 'best_conf' is not defined